# NVIDIA NIM と Google Kubernetes Engine でつくる高性能 AI 推論基盤
---


このNotebookでは、ローカライズされたNVIDIA NIM(NVIDIA Inference Microservice)とGKE(Google Kubernetes Engine)を使用してRAG (検索拡張生成)パイプラインを構築するデモンストレーションを行います。まず初めに、RAGの概要およびNVIDIA API Keyの設定、NIMイメージの取得とデプロイ、ローカルにデプロイされたNIMを使用するRAGアプリケーションの構築について説明します。

- GKE(Google Kubernetes Engine)の概要およびNVIDIA NIMのGKE経由でのデプロイの方法
- NIMモデルにアクセスするためのNVIDIA APIキーの設定方法
- RAGのコンセプトの理解
- ウェブリンクからコンテンツを取得し、コンテンツをテキストチャンクに分割し、エンベッディングを作成し、ベクターストアでローカルに保存する方法
- ベクトルストアから埋め込みモデルをロードし、NVIDIA Endpointsを使用してRAGを構築する概念

## 前提条件
今回のハンズオンは [こちらの手順](https://github.com/GoogleCloudPlatform/gcp-getting-started-lab-jp/blob/master/gke-next25/tutorial.md) を実施し、Google Kubernetes Engineを使用して、LLMモデルおよび埋め込みモデルのNIMがデプロイされている事が前提となっております。
今回のハンズオンでは、LLMモデルのNIMと埋め込みモデルのNIMがそれぞれNVIDIA L4を1枚使う想定で作成されています。

### 事前準備
デプロイした LLM のサービスの外部 IP アドレスを環境変数に設定しておきます。

In [ ]:
import os
import getpass

os.environ["LLM_IP_ADDR"] =  #Swallow の サービス外部 IP を設定します
os.environ["EMB_IP_ADDRESS"] =  #Swallow の サービス外部 IP を設定します


以下のセルを実行し、今回のハンズオンに必要なPythonライブラリのインストールを行います。

In [ ]:
!pip install -r gke_handson_requirements.txt

## NVIDIA NIMとは?

NVIDIA NIMは、生成AIのデプロイを加速する為の推論向けマイクロサービスです。クラウド、データセンター、ワークステーションなど、あらゆる環境にデプロイできるように設計されています。業界標準(OpenAI APIなど)のAPI互換のAPIを提供しており、既存のコードのエンドポイントをNVIDIA NIMで起動したエンドポイントに変更するだけで簡単にお使いいただく事が可能です。
今回のハンズオンでは、Google GKEを用い、LLMモデルのNIMと埋め込みモデルのNIMをデプロイし、RAGアプリケーションの構築を行います。

### NVIDIA API KEYの取得
NVIDIA NIMは、[NVIDIA API Catalog](https://build.nvidia.com/explore/discover)で利用可能な、使いやすいオープンAPIを介して素早くアクセスできます。これは、オンラインで幅広いマイクロサービスにアクセスするためのプラットフォームです。NVIDIA NIMを使い始めるには、登録が必要な`NVIDIA API Key`が必要です。以下のスクリーンショットに示すように、`ログインボタンをクリックしてメールアドレスを入力し`、プロセスに従って残りの情報登録するか、[NVIDIA NGC](https://ngc.nvidia.com/signin)の登録(*アカウント名をクリック -> セットアップ -> パーソナルキーの生成*)からAPIキーを生成してください。プロセスが完了したら、API Keyを将来の使用のためにアクセスできる場所に保存してください。APIキーのサンプルは、`nvapi-`で始まり、アンダースコア `_` を含む64文字です。

すでにアカウントをお持ちの方は、この手順に従って`NVIDIA API KEY`を取得してください：

- [ここ](https://build.nvidia.com/explore/discover)からあなたのアカウントでログイン.
- ご希望のモデルをクリックしてください。
- [Input]で[Python]タブを選択し、[Get API Key]をクリックし、[Generate Key]をクリックします。
- 生成されたキーをコピーし、NVIDIA_API_KEYとして保存します。そこからエンドポイントにアクセスできるはずです。

<div style="text-align: center;">
  <!--<img src="imgs/builder_catalog.jpg" style="width: 800px; height: auto;">-->
  <img src="imgs/nim-catalog.png" style="width: 900px; height: auto;">
</div>

### クイックテストの開始

### Llama 3.1 Swallow NIM の動作確認
NIMが稼働していることを2つの方法で素早くテストできます:
- LangChain NVIDIA Endpoints
- 単純なOpenAI完了リクエスト

**パラメータの説明:**
- **base_url**:  NVIDIA NIM の docker イメージがデプロイされているURL
- **model**: デプロイされた NVIDIA NIM モデルの名前
- **temperature**: サンプリングのランダム性を調整する。温度を下げると、高い確率で単語が選択される可能性が高くなります。
- **top_p**: モデルがどの程度決定論的かを制御します。正確で事実に基づいた回答を求める場合は、この値を低く保ちます。より多様な回答を求める場合は、値を大きくします
- **max_tokens**: 生成される出力トークンの最大数


以下のセルを実行しLLMのNIMのエンドポイントの動作確認を行いましょう。

In [ ]:
!curl -s -X POST "http://${LLM_IP_ADDR}:8000/v1/chat/completions" \
    -H "accept: application/json" \
    -H "Content-Type: application/json" \
    -d '{ \
        "model": "llama-3.1-swallow-8b", \
        "messages": [ \
            { \
                "role": "user", \
                "content": "日本の首都はどこですか？" \
            } \
        ], \
        "temperature": 0.7, \
        "max_tokens": 50 \
    }' \
    | jq

エラーが出力された場合は、しばらく待ってから上記のセルを再実行してください。エラーは、NIMコンテナが完全に立ち上がっていないことが原因かもしれません。

### llama-3.2-nv-embedqa LLM の動作確認
次に埋め込みモデルのNIMについても、以下のセルを実行してテストしましょう。

In [ ]:
!curl -s -X POST "http://${EMB_IP_ADDRESS}:8000/v1/embeddings" \
    -H "accept: application/json" \
    -H "Content-Type: application/json" \
    -d '{ \
        "model": "/model", \
        "input": [ "NVIDIAの新しく出たGPUには何がありますか？名前を教えてください。" ], \
        "input_type": "query" \
    }' \
    | jq

## RAG(検索拡張生成)

### RAGとは?

RAG（検索拡張生成）は、外部ソースから取得した事実を用いて生成AIモデルの精度と信頼性を向上させる技術です。
事前に学習された大規模な言語モデル(LLM)は、そのパラメータに事実知識を格納することが示されており、ダウンストリームの自然言語処理タスクでファインチューニングを行うと、最先端の結果を達成します。しかし、知識にアクセスして正確に操作する能力はまだ限定的であるため、知識集約的なタスクでは、タスクに特化したアーキテクチャに比べて性能が劣ります。さらに、その決定に実証性を与え、世界知識を更新することは、依然として未解決の研究課題です。詳しくは[こちら](https://arxiv.org/pdf/2005.11401)をご確認下さい。 
 

#### RAGはどのように役立つのか？
RAGは、大規模言語モデル（LLM）の強力なセマンティック機能と、メタデータ、テキスト、画像、ビデオ、表、グラフなどを含むデータセットとの接続を支援するアーキテクチャです。

<div style="text-align: center;">
    <img src="https://www.nvidia.com/en-in/glossary/retrieval-augmented-generation/_jcr_content/root/responsivegrid/nv_container_6840787/nv_image.coreimg.100.630.png/1714589610702/rag-diagram-1920x1080.png" alt="RAG Architecture" style="width: 80%; max-width: 600px;"> <br>
    RAG Architecture
</div>

### RAG コンポーネント

RAGアプリケーションは`Retriever` と ` Generator`の2つの主要コンポーネントから構成されています: . `Retriever` (RET) コンポーネントは、ユーザからの問い合わせに応答して、保存されているデータセットから適切な情報を抽出します。続いて、` Generator`(GEN)コンポーネントは、ユーザーからの問い合わせと検索された情報を利用して応答を生成します。

RAGシステムのアーキテクチャは、主に以下の要素で構成されています:
- **RET**
    - 埋め込みモデル: 入力データを密なベクトル表現に変換するバイエンコーダーネットワーク
    - ベクトルストア: 高次元ベクトル埋め込みを格納し、クエリするために最適化されたデータベース.
    - *リ*-ランキング (オプショナル): クエリとの関連性に基づいて検索された情報に優先順位を付けるクロスエンコーダ

- **GEN**
    - 大規模言語モデル (LLM): 膨大なテキストデータで訓練されたニューラルネットワークで、人間のような応答を生成することができます。
    - ガードレール (オプショナル): 生成された出力が事前に定義された制約や品質基準に準拠していることを保証するために実装されたメカニズムです。

堅牢なRAGアプリケーションには、以下に関して優れている必要があります：
- `Retriever`パイプライン
- ` Generator`モデル
- 両者のリンク

ラボでは、これらのコンポーネントを1つずつインスタンス化し、それらをまとめます。

## LangChainの概要

[LangChain](https://python.langchain.com/v0.2/docs/introduction/)は、大規模言語モデル(LLM)を使用するアプリケーションの開発を簡素化するために設計された強力なフレームワークです。
RAGアプリケーションのために、LangChainは以下を提供します:

- 様々なファイルタイプのドキュメントローダー
- ドキュメントを分割するテキスト分割機能
- テキストをベクトル表現に変換する埋め込み
- 効率的な類似検索のためのベクトルストア
- 関連情報を取得するための`Retriever`メソッド
- クエリと応答を構造化するプロンプトテンプレート

### LLM AI Endpoints

`langchain-nvidia-ai-endpoints`パッケージには、NVIDIA NIM推論マイクロサービス上のモデルでアプリケーションを構築するLangChain統合が含まれています。[ChatNVIDIA](https://python.langchain.com/v0.2/docs/integrations/chat/nvidia_ai_endpoints/)クラスは `langchain_nvidia_ai_endpoints` の一部で、チャットアプリケーションのためのNVIDIA NIMへのアクセスや、ホストまたはローカルにデプロイされたマイクロサービスへの接続を可能にします。
以下では、`meta/llama3-8b-instruct`を使った例を示します。各モデルはカタログにあるモデル名キーで一意に識別されます（例：meta/llama3-8b-instruct、meta/llama-3.1-70b-instruct）。

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = ChatNVIDIA(base_url="http://{}:8000/v1".format(os.environ['LLM_IP_ADDR']), model="llama-3.1-swallow-8b", temperature=0.1, max_tokens=1000, top_p=1.0)

In [ ]:
result = llm.invoke("RAGとは何ですか?")
result

**予想される出力**:
```
AIMessage(content='RAGとは、Risk Assessment and Governance（リスク評価とガバナンス）の略称であり、一般的に、組織におけるリスク管理とガバナンスのプロセスを指します。組織は、RAGを通じて、ビジネス、財務、運用などに関するリスクを特定、評価、優先順位付けし、適切な対策を講じることで、リスクを管理します。', additional_kwargs={}, response_metadata={'role': 'assistant', 'content': 'RAGとは、Risk Assessment and Governance（リスク評価とガバナンス）の略称であり、一般的に、組織におけるリスク管理とガバナンスのプロセスを指します。組織は、RAGを通じて、ビジネス、財務、運用などに関するリスクを特定、評価、優先順位付けし、適切な対策を講じることで、リスクを管理します。', 'token_usage': {'prompt_tokens': 16, 'total_tokens': 115, 'completion_tokens': 99}, 'finish_reason': 'stop', 'model_name': 'institute-of-science-tokyo/llama-3.1-swallow-8b-instruct-v0.1'}, id='run--28eba379-a97b-4cba-9b09-6e9905ab79f8-0', usage_metadata={'input_tokens': 16, 'output_tokens': 99, 'total_tokens': 115}, role='assistant')
```


以下のセルを実行すると、利用可能なモデルのリストにアクセスできます。

In [ ]:
ChatNVIDIA.get_available_models()

#### 回答内容の分析方法 

前のセルからの出力成分を解読してみましょう。

In [ ]:
import pprint
from IPython.display import display, Markdown, Latex

主な応答内容を表示します。

In [ ]:
display(Markdown(result.content))

`response_metadata` はその他の重要な詳細を示します。

In [ ]:
result.response_metadata

#### LLM 応答の Dictionary Keys

- `role`: レスポンダを識別する (例: 'assistant')
- `content`:  AI からの実際のテキストレスポンス
- `token_usage`:
  - `prompt_tokens`: 入力に含まれるトークンの数
  - `total_tokens`: インタラクションで使われたトークンの総数
  - `completion_tokens`: AI のレスポンスに含まれるトークン数
- `finish_reason`: AIがテキスト生成を停止した理由
- `model_name`: 使用するAIモデルを指定

In [ ]:
# Let's try asking a followup question
result = llm.invoke("それはどう便利なのですか?")
display(Markdown(result.content))

**注意:** LLMのステートレスな振る舞いを確認してみましょう。したがって、関連するすべてのコンテキスト、背景、クエリーを単一のプロンプトで渡すことが重要です。

## RAG アプリケーション

このセクションでは、NIMを使って簡単なRAGアプリケーションを作成します。このアプリケーションは、ウェブリンクからデータソースを受け取り、コンテンツを文書として読み込み、文書を分割し、埋め込みをインスタンス化し、ベクトルデータベースに格納します。さらに、要約用とチャット用の2つのLLMを使って[会話検索チェーン](https://python.langchain.com/v0.1/docs/modules/chains/#conversationalretrievalchain-with-streaming-to-stdout)を作成する。組み合わせたLLMの重要性は、複雑なシナリオにおいて全体的な結果を向上させることを支援することです。最後に、関連するクエリープロンプトを生成するための質問ジェネレーターを追加します。以下にその手順を示します：

- 必要なライブラリをすべてインポート
- ウェブリンクのデータソースを作成 
- ウェブリンクからHTMLファイルをロードする関数を作成
- embeddingsとdocument text splitterの作成
- LangChainからNVIDIA AI Endpointを使って埋め込みを生成し、オフラインのvector storeに保存
- vector storeから埋め込みをロードし、NVIDIA Endpointを使用してRAGを構築
- 2つのLLMを使って会話型検索チェーンを作る
- 質問ジェネレーターを追加して、関連するクエリー・プロンプトを生成する

#### ライブラリのインポート

In [ ]:
from langchain.chains import ConversationalRetrievalChain, LLMChain
from langchain.chains.conversational_retrieval.prompts import CONDENSE_QUESTION_PROMPT, QA_PROMPT
from langchain.chains.question_answering import load_qa_chain
from langchain.memory import ConversationBufferMemory
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

#### ウェブリンク・データソースの作成

お好みのウェブリンクを差し替えたり、追加したりすることができます。

In [ ]:
urls = [
"https://www.fsa.go.jp/news/r5/20230829/230829_main.pdf",
"https://www.tioj.or.jp/activity/pdf/190619_06.pdf",
"https://www.pmda.go.jp/files/000251332.pdf",
"https://www.jili.or.jp/files/research/zenkokujittai/pdf/r3/i-xvii.pdf",
"https://www.zenginkyo.or.jp/fileadmin/res/news/news350331_1.pdf"
]


## 実装

#### PDF ファイルを読み込む関数を作る

以下は、PDFファイルを読み込むためのヘルパー関数です。

In [ ]:
from pypdf import PdfReader
from io import BytesIO
import requests
import re

def clean_japanese_pdf_text(text: str) -> str:
    # 日本語文中の改行を削除
    text = re.sub(r'(?<=[ぁ-んァ-ヶ一-龥])\s*\n\s*(?=[ぁ-んァ-ヶ一-龥])', '', text)
    # 連続するスペースや全角スペースを1つにまとめる
    text = re.sub(r'[ 　]{2,}', ' ', text)
    # 各行の先頭空白を削除
    text = re.sub(r'^\s+', '', text, flags=re.MULTILINE)
    # 「..... 4」形式の目次行のページ番号を除去
    text = re.sub(r'\.{3,}\s*\d{1,3}', '', text)
    text = text.replace("\n", "")
    return text

def pdf_document_loader(url: str) -> str:
    """
    Loads and extracts cleaned text from a PDF at the given URL using `pypdf`.

    Args:
        url: The URL of the PDF.

    Returns:
        Cleaned text content of the PDF.
    """
    try:
        response = requests.get(url)
        response.raise_for_status()
        reader = PdfReader(BytesIO(response.content))
        text = ""
        for page in reader.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted
        return clean_japanese_pdf_text(text.strip())
    except Exception as e:
        print(f"Failed to load {url} due to: {e}")
        return ""



#### 埋め込みとDocument Text Splitterの作成

埋め込みを保存するパスを初期化し、`pdf_document_loader`関数を実行し、ドキュメントをテキストの塊に分割する関数を作ってみましょう。

In [ ]:
from tqdm import tqdm

def create_embeddings(embeddings_model,embedding_path: str = "./embed"):

    embedding_path = "./embed"
    print(f"Storing embeddings to {embedding_path}")

    documents = []
    for url in tqdm(urls):
        print(url)
        document = pdf_document_loader(url)
        #document = html_document_loader(url)
        documents.append(document)


    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=0,
        length_function=len,
    )
    print("Total documents:",len(documents))
    texts = text_splitter.create_documents(documents)
    print("Total texts:",len(texts))
    index_docs(embeddings_model,url, text_splitter, texts, embedding_path,)
    print("Generated embedding successfully")

#### LangChainからNVIDIA AI Endpointsを使って埋め込みを生成する

このセクションでは、LangChain用のNVIDIA AI Endpointsを使って埋め込みを生成し、将来の再利用のためにエンベッディングを`/embed`ディレクトリのオフラインベクタストアに保存する方法をデモします。

In [ ]:
embeddings_model = NVIDIAEmbeddings(base_url="http://{}:8000/v1".format(os.environ['EMB_IP_ADDRESS']), model="/model") # or use nvidia/nv-embedqa-e5-v5

以下では、ドキュメントページのコンテンツをループしてテキストとメタデータを拡張し、[FAISS](https://faiss.ai/index.html)を適用する `index_docs` 関数を作成します。埋め込みはローカルに保存されます。

In [ ]:
from typing import List, Union


def index_docs(embeddings_model, url: Union[str, bytes], splitter, documents: List[str], dest_embed_dir: str) -> None:
    """
    Split the documents into chunks and create embeddings for them.
    
    Args:
        embeddings_model: Model used for creating embeddings.
        url: Source url for the documents.
        splitter: Splitter used to split the documents.
        documents: List of documents whose embeddings need to be created.
        dest_embed_dir: Destination directory for embeddings.
    """
    texts = []
    metadatas = []

    for document in documents:
        chunk_texts = splitter.split_text(document.page_content)
        texts.extend(chunk_texts)
        metadatas.extend([document.metadata] * len(chunk_texts))

    if os.path.exists(dest_embed_dir):
        docsearch = FAISS.load_local(
            folder_path=dest_embed_dir, 
            embeddings=embeddings_model, 
            allow_dangerous_deserialization=True
        )
        docsearch.add_texts(texts, metadatas=metadatas)
    else:
        docsearch = FAISS.from_texts(texts, embedding=embeddings_model, metadatas=metadatas)

    docsearch.save_local(folder_path=dest_embed_dir)

#### ベクターストアからの埋め込みのロードとNVIDIA Endpointを使用したRAGの構築

次に、関数 `create_embeddings` を呼び出し、FAISSを使って[vector store](https://developer.nvidia.com/blog/accelerating-vector-search-fine-tuning-gpu-index-algorithms/)から文書を読み込む。ベクトルストアはembeddingsと呼ばれる高次元空間に関連情報を格納しています。

以下の2つのセルを実行してください。

In [ ]:
%%time
create_embeddings(embeddings_model=embeddings_model)

In [ ]:
# load Embed documents
embedding_path = "./embed/"
docsearch = FAISS.load_local(folder_path=embedding_path, embeddings=embeddings_model, allow_dangerous_deserialization=True)

### llama-3.1-swallow-8b-instructで会話型検索チェーンを作る

デプロイされたNIMは`http://0.0.0.0:8000`で稼働しているので、NIMの基本モデル`llama-3.1-swallow-8b-instruct`に基づいて[会話型検索チェーン](https://python.langchain.com/v0.1/docs/modules/chains/#conversationalretrievalchain-with-streaming-to-stdout)を作成する。

In [ ]:
# llm = ChatNVIDIA(base_url="http://0.0.0.0:{}/v1".format(os.environ['CONTAINER_PORT']),
#                model="tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1", temperature=0.0, max_tokens=1000, top_p=1.0)

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

qa_prompt=QA_PROMPT

doc_chain = load_qa_chain(llm, chain_type="stuff", prompt=QA_PROMPT)

qa = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=docsearch.as_retriever(),
    chain_type="stuff",
    memory=memory,
    combine_docs_chain_kwargs={'prompt': qa_prompt},
)

### クエリによるテスト

In [ ]:
query = "世帯主が不意の事故により入院が必要になる場合の必要資金について、60～64歳及び65歳以上の夫婦が公的年金以 \
        外に必要とする月間生活資金と比較してください。"
result = qa({"question": query})
print(result.get("answer"))

In [ ]:
query = "製造たばこの個包装及び中間包装に求められる識別マークの表示方法の条件について日本語で説明してください。"
result = qa({"question": query})
print(result.get("answer"))

---

## References

- https://developer.nvidia.com/blog/tips-for-building-a-rag-pipeline-with-nvidia-ai-langchain-ai-endpoints/
- https://nvidia.github.io/GenerativeAIExamples/latest/notebooks/05_RAG_for_HTML_docs_with_Langchain_NVIDIA_AI_Endpoints.html

## Licensing

Copyright © 2024 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.